# 05b - Boosting: AdaBoost dan Gradient Boosting
Materi Modul 5: meningkatkan dan menumpuk, gambaran umum AdaBoost dan peningkatan gradien, sintaks, lab.

**Boosting** melatih model **berurutan**; tiap model baru fokus memperbaiki kesalahan model sebelumnya. Menurunkan **bias**.
- **AdaBoost**: menaikkan bobot sampel yang salah diklasifikasi
- **Gradient Boosting**: model baru menebak *residual / gradien* dari model sebelumnya

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 1. AdaBoost (weak learner = pohon sangat dangkal / stump)

In [ ]:
stump = DecisionTreeClassifier(max_depth=1)
print('Satu stump | akurasi test =', round(stump.fit(X_train, y_train).score(X_test, y_test), 4))

ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=100, learning_rate=1.0, random_state=42)
ada.fit(X_train, y_train)
print('AdaBoost   | akurasi test =', round(ada.score(X_test, y_test), 4))

## 2. Akurasi bertambah seiring jumlah model (`staged_predict`)

In [ ]:
from sklearn.metrics import accuracy_score

acc_tr = [accuracy_score(y_train, p) for p in ada.staged_predict(X_train)]
acc_te = [accuracy_score(y_test, p) for p in ada.staged_predict(X_test)]

plt.plot(acc_tr, label='Train'); plt.plot(acc_te, label='Test')
plt.xlabel('Jumlah model (iterasi)'); plt.ylabel('Akurasi'); plt.legend()
plt.title('AdaBoost per iterasi'); plt.show()

## 3. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
print('Gradient Boosting | akurasi train =', round(gb.score(X_train, y_train), 4))
print('Gradient Boosting | akurasi test  =', round(gb.score(X_test, y_test), 4))

## 4. `learning_rate` vs `n_estimators`
Learning rate kecil butuh lebih banyak model, tapi biasanya lebih akurat dan lebih tahan overfit.

In [ ]:
print('learning_rate | n_estimators | CV')
for lr, n in [(1.0, 50), (0.5, 100), (0.1, 100), (0.1, 300), (0.01, 300)]:
    m = GradientBoostingClassifier(learning_rate=lr, n_estimators=n, random_state=42)
    print(f'{lr:<13} | {n:<12} | {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')

## 5. Pengaruh `max_depth` dan `subsample` (stochastic gradient boosting)

In [ ]:
for md in [1, 2, 3, 5]:
    m = GradientBoostingClassifier(max_depth=md, random_state=42)
    print(f'max_depth={md} | CV = {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')
print()
for ss in [1.0, 0.8, 0.5]:
    m = GradientBoostingClassifier(subsample=ss, random_state=42)
    print(f'subsample={ss} | CV = {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')

## 6. Bandingkan dengan Random Forest

In [ ]:
modelmodel = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}
for nama, m in modelmodel.items():
    print(f'{nama:18s} | CV = {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')

## Bagging vs Boosting
| | Bagging / RF | Boosting |
|---|---|---|
| Cara latih | Paralel, independen | Berurutan, saling memperbaiki |
| Menurunkan | Varians | Bias (dan varians) |
| Risiko overfit | Rendah | Lebih tinggi bila terlalu banyak iterasi |

## Catatan
- (cara kerja AdaBoost vs Gradient Boosting, peran learning_rate, dll.)